[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/01_wgcna.ipynb)

# R1-Q2 Week 2 — Build per-tissue co-expression networks

This notebook builds two co-expression networks — one for shoot, one for root — using PyWGCNA and the parameters you pre-committed in Week 1.

By the end of this notebook you will have:

- Two per-tissue WGCNA networks built with your pre-committed parameters (`shoot_wgcna.pkl`, `root_wgcna.pkl`).
- Soft-thresholding power diagnostics and module-structure inspection plots for each tissue.
- A `network_summary.json` (per-tissue gene count, module count, soft power used, key parameters) ready as Week 2 EQ#2 input.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

#!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory,
# and load the pre-commit you wrote in Notebook 0.
import json
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

precommit_path = OUTPUT_DIR / "precommit.json"
try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find your pre-commit file.\n"
        f"  Expected at: {precommit_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes `precommit.json` to this location.\n"
    ) from None

print(f"Loaded pre-commit with {len(precommit)} top-level entries.")

## 1. Load the filtered matrices and confirm your pre-committed parameters

In `00_question_orientation.ipynb` you produced two filtered expression matrices — one for shoot samples, one for root — and wrote your network-construction choices to `precommit.json`. This section loads both back into memory and prints your committed choices so they're visible alongside the code that will use them.

This is a deliberate beat, not bookkeeping. The whole point of pre-committing parameters in writing is that you can see them when the analysis runs. If something in the printed summary surprises you, stop and re-read what you wrote in N0 before continuing.

In [ ]:
# Load the filtered shoot and root matrices written by Notebook 0.
import pandas as pd

shoot_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_path  = OUTPUT_DIR / "root_filtered.parquet"

try:
    shoot = pd.read_parquet(shoot_path)
    root  = pd.read_parquet(root_path)
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"\nCould not find a filtered matrix.\n"
        f"  Expected: {shoot_path}\n"
        f"            {root_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes both files to this location.\n"
    ) from None

In [ ]:
# Confirm matrix shapes. Each matrix should be samples (rows) by genes (columns).
print("Shoot matrix:")
print(f"  {shoot.shape[0]} genes  ×  {shoot.shape[1]} samples")
print()
print("Root matrix:")
print(f"  {root.shape[0]} genes  ×  {root.shape[1]} samples")

# After the print:
assert shoot.shape[0] > shoot.shape[1], (
    f"Expected genes × samples (more rows than columns); got {shoot.shape}. "
    "Has N0's on-disk convention changed?"
)
assert root.shape[0] > root.shape[1], (
    f"Expected genes × samples (more rows than columns); got {root.shape}. "
    "Has N0's on-disk convention changed?"
)

Both matrices should have **gene counts in the same ballpark**. N0's MAD filter
used the same percentile cutoff per tissue but applied it independently to each —
exact equality is coincidence, not guarantee. Sample counts depend on which
AtGenExpress samples N0 retained per tissue; in practice these are matched (both
tissues come from the same plants), so equal or near-equal sample counts are
expected. If either shape looks far off from what you expect — for example, an
order-of-magnitude difference between tissues, or sample counts under 50 — go
back to N0 before continuing.

In [ ]:
# Echo the pre-committed network-construction choices you wrote in N0.
nc = precommit["network_construction"]
p  = nc["params"]

print("Pre-committed network-construction choices")
print("-" * 44)
print(f"Method:                {nc['method']}")
print(f"Network type:          {p['networkType']}")
print(f"TOM type:              {p['TOMType']}")
print(f"R² cutoff (scale-free): {p['RsquaredCut']}")
print(f"Minimum module size:   {p['minModuleSize']}")
print(f"Module merge threshold: {p['MEDissThres']}")
print()
print("Reasoning:")
print(nc["reasoning"])

These are the choices you'll feed PyWGCNA in section 2. The soft-thresholding power isn't fixed yet — PyWGCNA will scan a default range of candidate powers and pick the lowest one that achieves scale-free topology at the R² cutoff you pre-committed (0.9). Section 2 will record the chosen power in both `precommit.json` (under `network_construction.applied`) and `network_summary.json` (for Week 2 EQ#2).

With matrices loaded and parameters confirmed, you're ready to build the networks.

## 2. Build the per-tissue co-expression networks

Now you'll build two co-expression networks using PyWGCNA — one for shoot samples, one for root samples — feeding each the parameters you pre-committed.

PyWGCNA's `findModules()` does a lot of work in a single call: it scans candidate soft-thresholding powers, picks the one that achieves your R² cutoff, builds the adjacency matrix and topological overlap matrix, detects modules via dynamic tree cut, and merges modules whose eigengenes are similar enough. Expect each tissue to take a few minutes on Colab; you'll see progress messages as each step runs, plus a diagnostic plot showing the soft-power selection.

The shoot and root networks are built independently. There's no shared state between them.

In [ ]:
# PyWGCNA expects samples in rows, genes in columns.
# Our parquet files store genes in rows (the more common convention upstream),
# so transpose here before handing the matrices to PyWGCNA.
shoot_expr = shoot.T
root_expr  = root.T

print(f"Shoot expression matrix for PyWGCNA: {shoot_expr.shape[0]} samples × {shoot_expr.shape[1]} genes")
print(f"Root expression matrix for PyWGCNA:  {root_expr.shape[0]} samples × {root_expr.shape[1]} genes")

In [ ]:
# Build the shoot network.
import PyWGCNA

shoot_wgcna = PyWGCNA.WGCNA(
    name="shoot",
    species="arabidopsis thaliana",
    geneExp=shoot_expr,
    outputPath=str(OUTPUT_DIR) + "/",
    networkType=p["networkType"],
    TOMType=p["TOMType"],
    RsquaredCut=p["RsquaredCut"],
    minModuleSize=p["minModuleSize"],
    MEDissThres=p["MEDissThres"],
    save=False,  # we save deliberately in S4, not as a side effect here
)

shoot_wgcna.findModules()

In [ ]:
# Build the root network.
root_wgcna = PyWGCNA.WGCNA(
    name="root",
    species="arabidopsis thaliana",
    geneExp=root_expr,
    outputPath=str(OUTPUT_DIR) + "/",
    networkType=p["networkType"],
    TOMType=p["TOMType"],
    RsquaredCut=p["RsquaredCut"],
    minModuleSize=p["minModuleSize"],
    MEDissThres=p["MEDissThres"],
    save=False,
)

root_wgcna.findModules()

Both networks are built. Each WGCNA object now carries the soft-thresholding power that was selected, the adjacency matrix, the topological overlap matrix, the module assignments per gene, and the module eigengenes. The next section records the soft-power that PyWGCNA chose into your pre-commit file and into a fresh network_summary.json for Week 2 EQ#2.

## 3. Record what PyWGCNA did

You pre-committed your network-construction parameters in Notebook 0 — what kind of network, what cutoff, what minimum module size. PyWGCNA just ran the analysis and made some derived choices the pre-commit didn't fix: which soft-thresholding power to use, how many modules to extract. This section records those derived choices in two places.

The first is your pre-commit file (`precommit.json`), under a new `applied` block that sits alongside `params`. The structure becomes: "here's what I said I'd do, and here's what got done." If those two halves ever drift apart, you've made an undocumented choice and the pre-commit's job — making your analytical decisions visible — has failed.

The second is a fresh `network_summary.json` — the small, focused artifact that's part of your Week 2 EQ#2 deliverable. It's a portable summary of what came out of N1.

In [ ]:
# Extract per-tissue results from the WGCNA objects.
def summarize_network(wgcna):
    """Return a dict of the derived choices PyWGCNA made for one tissue."""
    return {
        "soft_thresholding_power": int(wgcna.power),
        "n_samples": int(wgcna.geneExpr.shape[0]),  # PyWGCNA stores samples in dim 0 (obs)
        "n_genes":   int(wgcna.geneExpr.shape[1]),  # genes in dim 1 (var)
        "n_modules_total": int(wgcna.datME.shape[1]),
        "module_names": list(wgcna.getModuleName()),
    }

shoot_summary = summarize_network(shoot_wgcna)
root_summary  = summarize_network(root_wgcna)

print("Shoot:")
for k, v in shoot_summary.items():
    print(f"  {k}: {v}")
print("\nRoot:")
for k, v in root_summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Update precommit.json with the applied block.
precommit["network_construction"]["applied"] = {
    "shoot": shoot_summary,
    "root":  root_summary,
}

with open(precommit_path, "w") as f:
    json.dump(precommit, f, indent=2)

print(f"Updated {precommit_path}")
print(f"  added: network_construction.applied.shoot")
print(f"  added: network_construction.applied.root")

In [ ]:
# Write a fresh network_summary.json for Week 2 EQ#2.
network_summary = {
    "shoot": shoot_summary,
    "root":  root_summary,
    "common_params": {
        "networkType":   precommit["network_construction"]["params"]["networkType"],
        "TOMType":       precommit["network_construction"]["params"]["TOMType"],
        "RsquaredCut":   precommit["network_construction"]["params"]["RsquaredCut"],
        "minModuleSize": precommit["network_construction"]["params"]["minModuleSize"],
        "MEDissThres":   precommit["network_construction"]["params"]["MEDissThres"],
    },
}

network_summary_path = OUTPUT_DIR / "network_summary.json"
with open(network_summary_path, "w") as f:
    json.dump(network_summary, f, indent=2)

print(f"Wrote {network_summary_path}")

A few things to notice in the numbers above.

For shoot, PyWGCNA selected soft-thresholding power **11**. The scale-free fit
climbed smoothly from R² ≈ 0.02 at power 1, crossed your pre-committed R² ≥ 0.9
cutoff at power 11 (R² = 0.908), and continued climbing toward 0.95 at higher
powers. Mean connectivity at the selected power dropped to ~1.5 — well inside
the textbook-healthy range. The selection rule worked as documented: PyWGCNA
picked the first power to clear the cutoff.

For root, PyWGCNA selected power **6**. Root's scale-free fit reached the 0.9
cutoff much faster than shoot's — R² was already 0.77 at power 2 and 0.91 by
power 6. The curve then plateaus through power 19 rather than continuing to
climb. Different powers for different tissues is the expected outcome of
building separate per-tissue networks (it's why R1-Q2 does so in the first
place); the difference reflects real structural differences between root and
shoot expression networks.

Both tissues produced **9 modules** after dynamic tree cut and eigengene-distance
merging. The shared count is coincidental — shoot's path was 11 → 10 → 9 over
two merge rounds, root's was 14 → 12 → 11 → 9 over three. The module names
differ substantially between tissues (only `black`, `darkgrey`, `dimgrey`,
`gainsboro`, and `lightcoral` appear in both), which is what we'd expect when
the same algorithm finds biologically distinct co-expression structure in two
different tissues.

The `applied` block in `precommit.json` records what PyWGCNA actually did for
each tissue alongside the parameters you pre-committed. A reader of the artifact
can see at a glance which decisions PyWGCNA made (soft power, module count) and
which you locked in advance (network type, TOM type, R² cutoff, minimum module
size, merge threshold).

## 4. Save the WGCNA objects to disk

Each WGCNA object now holds everything Notebook 2 needs to find hubs: the
soft-thresholding power, the topological overlap matrix, the per-gene
module assignments, and the module eigengenes. Saving them as pickles
means N2 loads them as a single artifact per tissue rather than rebuilding
networks from scratch.

PyWGCNA's `saveWGCNA()` writes to the `outputPath` set at construction
time (Section 2) using `{name}.p` as the filename — so these will land at
`shoot.p` and `root.p` inside your R1-Q2 output directory.

In [ ]:
# Save both WGCNA objects to disk for Notebook 2 to load.
shoot_wgcna.saveWGCNA()
root_wgcna.saveWGCNA()

In [ ]:
# Round-trip verification: read back what we just saved and confirm a few
# stable attributes match what's currently in memory. saveWGCNA() doesn't
# return a success signal, so this is the only way to know the pickle is
# both present and loadable.
import pickle

shoot_pickle_path = OUTPUT_DIR / "shoot.p"
root_pickle_path  = OUTPUT_DIR / "root.p"

assert shoot_pickle_path.exists(), f"Expected {shoot_pickle_path} after saveWGCNA()."
assert root_pickle_path.exists(),  f"Expected {root_pickle_path} after saveWGCNA()."

with open(shoot_pickle_path, "rb") as f:
    shoot_reloaded = pickle.load(f)
with open(root_pickle_path, "rb") as f:
    root_reloaded = pickle.load(f)

# Spot-check stable scalar attributes. Full WGCNA equality is impractical
# because the objects carry AnnData, DataFrames, and numpy arrays.

# Shoot
assert shoot_reloaded.name == shoot_wgcna.name
assert int(shoot_reloaded.power) == int(shoot_wgcna.power)
assert shoot_reloaded.geneExpr.shape == shoot_wgcna.geneExpr.shape
assert sorted(shoot_reloaded.getModuleName()) == sorted(shoot_wgcna.getModuleName())
print(f"shoot: round-trip verified "
      f"(name={shoot_reloaded.name}, power={shoot_reloaded.power}, "
      f"shape={shoot_reloaded.geneExpr.shape}, "
      f"{len(shoot_reloaded.getModuleName())} modules)")

# Root
assert root_reloaded.name == root_wgcna.name
assert int(root_reloaded.power) == int(root_wgcna.power)
assert root_reloaded.geneExpr.shape == root_wgcna.geneExpr.shape
assert sorted(root_reloaded.getModuleName()) == sorted(root_wgcna.getModuleName())
print(f"root: round-trip verified "
      f"(name={root_reloaded.name}, power={root_reloaded.power}, "
      f"shape={root_reloaded.geneExpr.shape}, "
      f"{len(root_reloaded.getModuleName())} modules)")

Both WGCNA objects are saved and round-trip verified. N2 picks them up via
`pickle.load(open(OUTPUT_DIR / "shoot.p", "rb"))` (same for root) for hub
identification.

The next section inspects module structure — module size distribution,
the eigengene dendrograms PyWGCNA already produced, and module–eigengene
correlations against stress conditions in the attached metadata.

## 5. Inspect module structure

The networks are built and saved. This section surfaces what's *in* them:
which genes ended up where, how the modules are structured relative to
each other, and which modules correlate with which stress conditions in
the AtGenExpress sample metadata. None of this changes the analysis —
it's read-only inspection of what's already been computed, organized to
feed your Week 2 EQ#2 writeup and to set up N2's hub identification.

### 5.1) Attach sample metadata

`iri.atgenexpress_metadata()` returns a DataFrame indexed by GSM accession
with columns for stress condition, tissue, timepoint, replicate, and
source GSE. Attaching it to each WGCNA object's `obs` axis makes those
columns available for the per-condition correlation heatmap in 5.4.

In [ ]:
# Load the full AtGenExpress sample metadata. Both tissues share the
# same metadata source; filtering to per-tissue subsets happens below.
metadata = iri.atgenexpress_metadata()

print(f"Metadata: {metadata.shape[0]} samples × {metadata.shape[1]} columns")
print(f"Columns:  {list(metadata.columns)}")
print(f"\nFirst three samples:")
print(metadata.head(3))

In [ ]:
# Filter metadata to each tissue's samples, then attach to the corresponding
# WGCNA object via updateSampleInfo(). The filter is for documentation —
# updateSampleInfo() would drop unmatched rows on its own.

# Shoot
shoot_metadata = metadata[metadata.index.isin(shoot_wgcna.datExpr.obs.index)]
shoot_wgcna.updateSampleInfo(sampleInfo=shoot_metadata)

# Root
root_metadata = metadata[metadata.index.isin(root_wgcna.datExpr.obs.index)]
root_wgcna.updateSampleInfo(sampleInfo=root_metadata)

print(f"shoot: attached {shoot_metadata.shape[0]} rows of metadata")
print(f"root:  attached {root_metadata.shape[0]} rows of metadata")

In [ ]:
# Verify metadata landed on the sample axis (obs). The previous run's
# orientation bug caused updateSampleInfo() to silently write all-NaN
# to the gene axis; this check would catch a recurrence immediately.

# Shoot
shoot_obs = shoot_wgcna.datExpr.obs
assert all(c in shoot_obs.columns for c in metadata.columns), \
    "Some metadata columns did not appear in shoot's obs after updateSampleInfo()."
n_complete = shoot_obs[metadata.columns.tolist()].notna().all(axis=1).sum()
print(f"shoot: {n_complete} / {len(shoot_obs)} samples have complete metadata")
print(f"  stress: {shoot_obs['stress'].value_counts().to_dict()}")
print(f"  tissue: {shoot_obs['tissue'].value_counts().to_dict()}")

# Root
root_obs = root_wgcna.datExpr.obs
assert all(c in root_obs.columns for c in metadata.columns), \
    "Some metadata columns did not appear in root's obs after updateSampleInfo()."
n_complete = root_obs[metadata.columns.tolist()].notna().all(axis=1).sum()
print(f"\nroot:  {n_complete} / {len(root_obs)} samples have complete metadata")
print(f"  stress: {root_obs['stress'].value_counts().to_dict()}")
print(f"  tissue: {root_obs['tissue'].value_counts().to_dict()}")

### 5.2) Module size distribution

How genes distribute across the nine modules per tissue. A healthy WGCNA
result has a few larger modules and a longer tail of smaller ones — the
inverse pattern (one module swallowing most of the genes, the rest tiny)
suggests the dynamic tree cut couldn't find real substructure and was
forced to lump genes into a single bin.

The module colors WGCNA uses are arbitrary labels, not meaningful
encodings. Two modules with similar names in different tissues
(`black` in shoot vs. `black` in root) are *not* the same module —
color assignment is per-run. The bar colors below match the WGCNA color
labels purely so the visualization matches what you'll see in the
literature.

In [ ]:
# Module size distribution per tissue. Read directly from datExpr.var,
# not from getModuleName(), so any 'grey' unassigned-gene bin shows up
# even though PyWGCNA drops it from the eigengene table.

shoot_sizes = shoot_wgcna.datExpr.var['moduleColors'].value_counts().sort_values(ascending=False)
root_sizes  = root_wgcna.datExpr.var['moduleColors'].value_counts().sort_values(ascending=False)

print("Shoot module sizes:")
for color, size in shoot_sizes.items():
    print(f"  {color:<14} {size:>4} genes")
print(f"  {'TOTAL':<14} {shoot_sizes.sum():>4} genes")

print("\nRoot module sizes:")
for color, size in root_sizes.items():
    print(f"  {color:<14} {size:>4} genes")
print(f"  {'TOTAL':<14} {root_sizes.sum():>4} genes")

# Grey-bin check: are there genes labeled 'grey' (unassigned) that won't
# appear in the eigengene table or the heatmap in 5.4?
for label, sizes in [("shoot", shoot_sizes), ("root", root_sizes)]:
    grey_n = sizes.get('grey', 0)
    if grey_n > 0:
        print(f"\nNote: {label} has {grey_n} genes in the 'grey' unassigned bin "
              f"(no eigengene; will not appear in 5.4).")

In [ ]:
# Bar plot of module sizes per tissue. Bars colored by the WGCNA color
# label (which is a literal CSS color name); black outlines so very light
# colors stay visible.
import matplotlib.pyplot as plt

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(12, 5))

ax_shoot.barh(shoot_sizes.index[::-1], shoot_sizes.values[::-1],
              color=shoot_sizes.index[::-1], edgecolor='black', linewidth=0.6)
ax_shoot.set_title('Shoot module sizes')
ax_shoot.set_xlabel('genes per module')

ax_root.barh(root_sizes.index[::-1], root_sizes.values[::-1],
             color=root_sizes.index[::-1], edgecolor='black', linewidth=0.6)
ax_root.set_title('Root module sizes')
ax_root.set_xlabel('genes per module')

plt.tight_layout()
plt.show()

What to look for: the largest module per tissue holds the bulk of the
genes — that's expected and isn't itself a problem. What you don't want
to see is *only* a large module plus tiny modules of two or three genes
each (that would mean the algorithm failed to find substructure). Each
tissue here shows a graduated distribution, which is what a healthy
WGCNA result looks like.

### 5.3) How to read the eigengene dendrograms

Section 2's `findModules()` calls produced one eigengene dendrogram per
tissue and displayed them inline in the notebook output. Scroll back to
see them — this subsection is a viewing guide for those plots, not a
new analysis.

Each dendrogram clusters the *pre-merge* module eigengenes (so 11 leaves
for shoot, 14 for root — these are the dynamic-tree-cut modules before
the merge step). The dashed horizontal line marks `MEDissThres`, the
merge threshold you pre-committed at 0.2. Branches that fuse *below*
that line have eigengenes correlated enough to be considered one module;
they get merged and contribute together to the final module count. The
colored sub-trees in each plot show those below-threshold fusions.

For shoot, two fusions sit below the line (at eigengene distances ~0.04
and ~0.11), producing the 11 → 9 module count you see in the summary.
The remaining seven modules sit comfortably above the threshold —
nearest non-merge pair is around 0.30. Shoot's merge structure is
robust: a slightly different threshold wouldn't change the outcome much.

For root, four fusions sit below the threshold (the orange, green, red,
and purple sub-trees), producing 14 → 9 over three merge rounds. Root's
top split is deeper than shoot's (~1.47 vs ~1.28), and the modules
organize into two clear sub-clusters that may correspond to higher-order
biological grouping worth investigating in the heatmap below. **One
fusion sits near the knife-edge: the merge that joins one of the green
sub-tree's eigengenes happens at distance ~0.22, just above your 0.2
threshold.** If you had pre-committed `MEDissThres = 0.25`, root would
have ended at 8 modules rather than 9. This is a sensitivity worth
acknowledging in your EQ#2 writeup — the pre-commit framework's whole
point is to surface this kind of choice rather than hide it.

### 5.4) Module-eigengene × stress condition heatmap

For each module, how strongly does its eigengene track each of the nine
stress conditions in the metadata? A module that correlates strongly with
one specific stress is a candidate "stress-relevant module" for N2's hub
identification — those are the modules whose hub genes are most likely
to be biologically meaningful regulators of the stress response.

The heatmap is computed by one-hot encoding the stress column from the
attached metadata (one binary 0/1 column per condition), then computing
Pearson correlation between each module's eigengene (a 124-vector across
samples) and each one-hot column (also a 124-vector). Result: a 9 × 9
matrix per tissue.

In [ ]:
# Compute the module-eigengene × stress correlation matrix per tissue.

# Deterministic column order so shoot and root heatmaps align visually
# rather than being alphabetized differently by pandas.
stress_order = ['control', 'cold', 'drought', 'genotoxic', 'heat',
                'osmotic', 'salt', 'uv_b', 'wounding']

# Shoot
shoot_onehot = pd.get_dummies(shoot_wgcna.datExpr.obs['stress'])[stress_order].astype(float)
shoot_onehot = shoot_onehot.loc[shoot_wgcna.MEs.index]  # align to MEs row order
shoot_corr = pd.DataFrame({
    s: shoot_wgcna.MEs.corrwith(shoot_onehot[s]) for s in stress_order
})
shoot_corr.index = [c.removeprefix('ME') for c in shoot_corr.index]

# Root
root_onehot = pd.get_dummies(root_wgcna.datExpr.obs['stress'])[stress_order].astype(float)
root_onehot = root_onehot.loc[root_wgcna.MEs.index]
root_corr = pd.DataFrame({
    s: root_wgcna.MEs.corrwith(root_onehot[s]) for s in stress_order
})
root_corr.index = [c.removeprefix('ME') for c in root_corr.index]

print("Shoot module × stress correlations:")
print(shoot_corr.round(2))
print("\nRoot module × stress correlations:")
print(root_corr.round(2))

In [ ]:
# Side-by-side heatmaps. Diverging colormap centered at zero so positive
# and negative correlations are visually distinct; values annotated in
# each cell for precise reading.

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(16, 7))

for ax, corr, title in [
    (ax_shoot, shoot_corr, "Shoot: module-eigengene × stress correlation"),
    (ax_root,  root_corr,  "Root: module-eigengene × stress correlation"),
]:
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title(title)
    for i in range(len(corr.index)):
        for j in range(len(corr.columns)):
            v = corr.values[i, j]
            text_color = 'white' if abs(v) > 0.55 else 'black'
            ax.text(j, i, f"{v:.2f}", ha='center', va='center',
                    color=text_color, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

**What to look for.** Modules whose eigengenes correlate strongly with
a specific stress are the candidates for "stress-relevant" in N2. A
correlation of ±0.3 or higher (in either direction) is worth attention;
a correlation of ±0.5 or higher is a clear signal. A module that
correlates weakly with everything is a generic-cellular-process module
rather than a stress-response one — still real biology, just not what
R1-Q2 is about.

**Cross-tissue contrast worth noting in EQ#2.** Root's heatmap shows
stronger and more numerous strong-correlation cells than shoot's. Root
has multiple modules reaching ±0.5–0.7 across heat, salt, and osmotic;
shoot tops out around ±0.6 and several shoot modules show no
correlation stronger than ±0.3 with any stress. This is consistent with
shoot's module-size distribution being dominated by one very large
module (`lightcoral` at ~3,000 genes) that's too generic to track any
specific stress — its eigengene averages over too much heterogeneous
expression to correlate cleanly with anything. Root's modules are more
evenly sized and consequently more stress-specific.

**Two structural caveats worth knowing.** First, AtGenExpress's salt
and osmotic stress collections are closely related biologically (both
perturb water/ion balance) — you'll see modules that correlate with
both, often in the same direction, because the underlying response
shares regulators. Second, salt and osmotic drive more sample-level
variance than the other conditions (visible in N0b's PCA), which means
modules tracking those stresses may show inflated correlation magnitudes
relative to modules tracking heat, cold, drought, UV-B, wounding, or
genotoxic stress. **When you interpret your hubs in N2, account for this
when comparing correlation strengths across stresses.** A hub found in
a salt-correlated module at r = 0.5 isn't automatically more important
than a hub found in a cold-correlated module at r = 0.4; the salt
correlation had more variance available to surface.